In [1]:
!pip install pymoo

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 15.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.1/249.1 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.1/77.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.4/119.4 kB 7.7 MB/s eta 0:00:00
  Created wheel for grapheme: filename=grapheme-0.6.0-py3-none-any.whl size=210082 sha256=c4ba265783107a244d4fb68ecee169fe94970a77d77c83e8356a0a87a4cb4996
  Stored in directory: /root/.cache/pip/wheels/ee/3b/0b/1b865800e916d671a24028d884698674138632a83fdfad4926
Successfully built grapheme


In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.algorithms.moo.nsga3 import NSGA3
from pymoo.core.problem import Problem
from pymoo.decomposition.asf import ASF
from pymoo.optimize import minimize
from pymoo.termination import get_termination
from pymoo.util.ref_dirs import get_reference_directions

from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import OrdinalEncoder

# Load Data

In [4]:
# Copy the files to your Google Drive and set your paths here
df_features = pd.read_csv('/content/drive/MyDrive/Estudos/Datasets/pas_aerogerador_helon/X10.csv', header=None)
df_rmse = pd.read_csv('/content/drive/MyDrive/Estudos/Datasets/pas_aerogerador_helon/RMSE10.csv', header=None)
df_aic = pd.read_csv('/content/drive/MyDrive/Estudos/Datasets/pas_aerogerador_helon/AIC10.csv', header=None)
df_labels = pd.read_csv('/content/drive/MyDrive/Estudos/Datasets/pas_aerogerador_helon/df2save.csv', index_col=0)

In [5]:
X = df_features.to_numpy()
y = df_labels['case'].isin(['A','B','C','R']).to_numpy().squeeze()

In [6]:
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, shuffle=True)

groups = df_labels['temp'].to_numpy().squeeze()
X_train, X_test = X[groups >= 0], X[groups < 0],
y_train, y_test = y[groups >= 0], y[groups < 0]

# RandomForest Experiment

In [7]:
mdl = RandomForestClassifier(max_samples=0.5).fit(X_train, y_train)
y_pred = mdl.predict(X_test)

In [8]:
print(
    classification_report(
        y_test, y_pred
    )
)

              precision    recall  f1-score   support

       False       0.99      0.93      0.95       361
        True       0.90      0.98      0.94       240

    accuracy                           0.95       601
   macro avg       0.94      0.95      0.95       601
weighted avg       0.95      0.95      0.95       601



# Modified RandomForest with Multi-objective Ensemble Model Selection (RF-MOEMS)

In [84]:
class MOEMC(Problem):

    def __init__(self, y_all, y):
        super().__init__(n_var=len(y_all), n_obj=2, xl=0.0, xu=1.0)
        self.y_all = [2*yy-1.0 for yy in y_all]
        self.y = y

    def _evaluate(self, x, out, *args, **kwargs):
        errors = []
        complexities = []
        for xx in x:
            if sum(xx) == 0:
                errors.append(10)
                complexities.append(10)
                continue
            # xx = xx/sum(xx)
            # print(sum(xx), xx)
            y_out = [xp*out for out, xp in zip(self.y_all, xx)]
            y_pred = pd.DataFrame(y_out).mean().to_numpy() > 0.0
            complexity = sum(xx)/len(xx)
            complexities.append(complexity)
            error = 1 - classification_report(self.y, y_pred, output_dict=True)['accuracy']
            errors.append(error)
        out['F'] = np.column_stack([complexities, errors])
        # out['F'] = np.column_stack([errors, errors])

    def get_pareto_front(self, X):
        out = {}
        self._evaluate(X, out)
        return out['F']

In [88]:
class RFMOEMC():
    def __init__(self, n_classifiers=100, optimization_set_ratio=0.2):
        self.n_classifiers = n_classifiers
        self.opt_ratio = optimization_set_ratio
        self.rf = RandomForestClassifier(self.n_classifiers, max_samples=0.5)
        self.moemc = [[True for _ in range(n_classifiers)]]
        self.weights = 0

    def predict_all(self, X):
        y_out = []
        for i, dt in enumerate(self.rf.estimators_):
            y_out.append(dt.predict(X))
        return y_out

    def _moo(self, X, y):
        y_out_all = self.predict_all(X)
        prob = MOEMC(y_out_all, y)
        alg = NSGA3(pop_size=20, ref_dirs=get_reference_directions("uniform", 2, n_partitions=10))
        termination = get_termination("n_gen", 20)
        return minimize(prob, alg, termination, verbose=True)

    def _mcdm(self, res):
        weights = np.array([0.2, 0.8])
        decomp = ASF()
        return decomp.do(res.F, 1/weights).argmin()

    def _fit_moemc(self, X, y):
        pareto = self._moo(X, y)
        self.moemc = [[v for v in x] for x in pareto.X]
        self.weights = self.moemc[self._mcdm(pareto)]

    def fit(self, X_rf, y_rf, X_opt=None, y_opt=None):
        if X_opt is None or y_opt is None:
            X_rf, X_opt, y_rf, y_opt = train_test_split(X_rf, y_rf, test_size=self.opt_ratio, shuffle=True)
        self.rf.fit(X_rf, y_rf)
        self._fit_moemc(X_opt, y_opt)
        return self

    def _predict_moemc(self, X, weights):
        y_out = []
        for i, (dt, w) in enumerate(zip(self.rf.estimators_, weights)):
            y_out.append(w*(2*dt.predict(X)-1))
        return y_out

    def get_number_solutions(self):
        return len(self.moemc)

    def get_selected_solution(self):
        return self.selected

    def predict_weighted(self, X, weights):
        y_out_all = self._predict_moemc(X, weights)
        return pd.DataFrame(y_out_all).mean().to_numpy() > 0.0

    def predict(self, X):
        return self.predict_weighted(X, self.weights)

In [89]:
X_rf, X_opt, y_rf, y_opt = train_test_split(X_train, y_train, test_size=0.2, shuffle=True)
mdl = RFMOEMC().fit(X_rf, y_rf, X_opt, y_opt)
y_pred = mdl.predict(X_test)

n_gen  |  n_eval  | n_nds  |      eps      |   indicator  
     1 |       20 |      1 |             - |             -
     2 |       40 |      1 |  0.000000E+00 |             f
     3 |       60 |      1 |  0.0141613098 |         ideal
     4 |       80 |      1 |  0.0186563598 |         ideal
     5 |      100 |      1 |  0.0039726847 |         ideal
     6 |      120 |      1 |  0.000000E+00 |             f
     7 |      140 |      1 |  0.0028305845 |         ideal
     8 |      160 |      1 |  0.0013551638 |             f
     9 |      180 |      1 |  0.0050122076 |         ideal
    10 |      200 |      1 |  0.0236896707 |         ideal
    11 |      220 |      1 |  0.0033196165 |         ideal
    12 |      240 |      1 |  0.0051419435 |         ideal
    13 |      260 |      1 |  0.0021729140 |             f
    14 |      280 |      1 |  0.0267758826 |         ideal
    15 |      300 |      1 |  0.000000E+00 |             f
    16 |      320 |      1 |  0.0040896998 |         ide

In [90]:
print(
    classification_report(
        y_test, y_pred
    )
)

              precision    recall  f1-score   support

       False       0.98      0.93      0.95       361
        True       0.90      0.97      0.93       240

    accuracy                           0.95       601
   macro avg       0.94      0.95      0.94       601
weighted avg       0.95      0.95      0.95       601



In [ ]:
problem_test = MOEMC(mdl.predict_all(X_test), y_test)
problem_opt = MOEMC(mdl.predict_all(X_opt), y_opt)

In [ ]:
decision_space = mdl.moems
pareto_front_test = problem_test.get_pareto_front(decision_space)
pareto_front = problem_opt.get_pareto_front(decision_space)

In [ ]:
fig, axs = plt.subplots(1,2, figsize=(15,5))

X_2d = PCA(n_components=2).fit_transform(np.array(decision_space))

for i in range(len(pareto_front)):
    axs[0].scatter(pareto_front[i,0], pareto_front[i,1])
    axs[1].scatter(X_2d[i,0], X_2d[i,1])

axs[0].set_title('Pareto Front')
axs[0].set_xlabel('Complexity')
axs[0].set_ylabel('Error')
axs[1].set_title('Decision Space (PCA)')
axs[1].set_xlabel('PCA1')
axs[1].set_ylabel('PAC2')
plt.show()

In [ ]:
fig, axs = plt.subplots(1,2, figsize=(15,5))

X_2d = PCA(n_components=2).fit_transform(np.array(decision_space))

for i in range(len(pareto_front_test)):
    axs[0].scatter(pareto_front_test[i,0], pareto_front_test[i,1])
    axs[1].scatter(X_2d[i,0], X_2d[i,1])

axs[0].set_title('Pareto Front')
axs[0].set_xlabel('Complexity')
axs[0].set_ylabel('Error')
axs[1].set_title('Decision Space (PCA)')
axs[1].set_xlabel('PCA1')
axs[1].set_ylabel('PAC2')
plt.show()